# 01 — Empathy

**Task.** Binary classification: given a feedback email written to a colleague named Jonathan about his work on the "Beta project," decide whether the email demonstrates empathy (1) or not (0). The score is plain accuracy on a held-out test set.

**What the winners got.** Akben .608 · PAID .580 · Hungry Llama .560 · Wonderlic .488.

**What this notebook does.** Walk through three approaches: a naive baseline, a reconstruction of the PAID Team's auto-reasoning approach, and the unified-harness alternative this repo ships. Then discuss why Akben's Elo-rating idea beat both.

For background on each team's full approach, see `docs/WINNERS_SYNTHESIS.md`.

## Setup

This notebook expects the input text files in `data/`. See `data/README.md` for what's needed and where to get it. If the files are missing, the cells will print clear messages and skip — the prompt-construction and parsing logic is still inspectable.

In [ ]:
import sys
from pathlib import Path

# Make src importable from the notebooks/ directory
sys.path.insert(0, str(Path.cwd().parent))

import json
import csv
from src.adapters import EmpathyAdapter
from src.harness import Harness, CallSpec, mode_reducer
from src.examples import pick_random, pick_similar
from src.scoring import accuracy
from src.run import load_csv, _load_labels, DATA_DIR

# Sanity check what data we have
print(f"Data directory: {DATA_DIR.resolve()}")
print(f"Labels file (dev): {(DATA_DIR / 'dev.csv').exists()}")
print(f"Empathy inputs (train): {(DATA_DIR / 'empathy_train.csv').exists()}")
print(f"Empathy inputs (dev):   {(DATA_DIR / 'empathy_val_public.csv').exists() or (DATA_DIR / 'empathy_dev_inputs.csv').exists()}")
print(f"Empathy inputs (test):  {(DATA_DIR / 'empathy_test_public.csv').exists() or (DATA_DIR / 'empathy_test_inputs.csv').exists()}")

## The data

The empathy training set (from the competition release) contains roughly 200 emails with binary empathy labels. Each email is written to "Jonathan" about his work on the "Beta project" — the prompt templates were standardized so all candidates wrote about the same scenario. Here's a representative example from the Akben deck:

> *"Hi Jonathan, I hope this message finds you well. I hear things are going well with the Beta project. That said, Terry mentioned that there were some issues with the reports. From what I understand, they would like them to be more concise and straight to the point... I recommend you reach out to Terry so you both could review in detail one of the reports he submits. This should help you help you align to their expectations. Additionally, I'd be happy to review the reports before you send them off to Terry and provide my feedback. I know this project is important to you, so please let me know how this meeting goes and how else I can help. Regards, William"*

This is unambiguously empathetic. The interesting cases are the borderline ones — emails that are direct but polite, or technically supportive but tonally cold.

In [ ]:
# Load the empathy training set if available, otherwise show what we'd do
empathy_train = load_csv(DATA_DIR / "empathy_train.csv")
if empathy_train:
    print(f"Loaded {len(empathy_train)} training examples")
    print(f"Columns: {list(empathy_train[0].keys())}")
    print(f"\nLabel distribution: {sum(int(r['empathy']) for r in empathy_train)} positive / {len(empathy_train)} total")
    print(f"\nFirst example:")
    ex = empathy_train[0]
    print(f"  empathy = {ex['empathy']}")
    print(f"  text = {ex['text'][:200]}...")
else:
    print("Empathy training data not present.")
    print("Synthetic example for prompt inspection:")
    empathy_train = [
        {"_id": "sample1", "text": "Hi Jonathan, great work on the project!", "empathy": "1"},
        {"_id": "sample2", "text": "Jonathan, the reports are wrong. Fix them.", "empathy": "0"},
    ]

## Baseline — naive zero-shot

The simplest possible approach: ask GPT-4o "is this empathetic? 0 or 1" with no examples. This is the floor.

We don't run this against the test set; the point is just to see what the prompt looks like and confirm the harness round-trips. Expected accuracy: ~.50 (chance) to ~.55 (the model has some empathy-recognition prior from its training).

In [ ]:
# Build a zero-shot empathy prompt manually (not via the adapter, so it's inspectable)
sample_row = {"text": "Hi Jonathan, your reports need work. Please be more concise."}

baseline_messages = [
    {"role": "system", "content": "Decide whether the following workplace email demonstrates empathy. Respond with just 0 (no empathy) or 1 (empathy)."},
    {"role": "user", "content": f"Email: {sample_row['text']}"},
]
print(json.dumps(baseline_messages, indent=2))

## PAID Team approach reconstruction

PAID Team won the competition partly because of what they did on empathy — though Akben actually beat them on this specific task. PAID's empathy approach was:

1. **Step 1** — for every training email, ask GPT-4 to *generate a reason* why this email did or did not demonstrate empathy. Cache these reasons.
2. **Step 2** — for each test row, construct a long conversation history where each "training email + reason" pair becomes a user/assistant turn. End with the test row as a user turn and ask for the empathy label.

The key insight: GPT-4's own generated reasoning becomes the few-shot teaching signal. The model is calibrating against its own explanatory framework, not just labels.

Below is a reconstruction. The "Step 1" pass is expensive — one call per training row — so the implementation caches results.

In [ ]:
def step_1_generate_reasons(train_rows, harness, model="gpt-4o-2024-08-06"):
    '''For each training row, ask the model why the email did/did not demonstrate empathy.
    Returns a list of dicts with 'text', 'empathy', and 'reason' keys.
    '''
    out = []
    for row in train_rows:
        label_word = "did" if int(row["empathy"]) == 1 else "did not"
        prompt = (
            f"Here is a feedback email written to Jonathan about his work on the Beta project.\n\n"
            f"Email:\n###\n{row['text']}\n###\n\n"
            f"This email {label_word} demonstrate empathy. Provide a short answer (1-2 sentences) explaining why."
        )
        spec = CallSpec(
            messages=[{"role": "user", "content": prompt}],
            model=model,
            temperature=0.0,
        )
        # In a real run, this calls the API. With no API key set, it'll raise; that's fine for inspection.
        # reason = harness.call(spec)
        # For notebook walkthrough purposes, we skip the call and show the structure:
        reason = "<would call API here>"
        out.append({**row, "reason": reason})
    return out

# Show what one prompt looks like
sample_train = empathy_train[0] if empathy_train else {"text": "...", "empathy": "1"}
label_word = "did" if int(sample_train["empathy"]) == 1 else "did not"
print(f"Step 1 prompt for training example {sample_train.get('_id', 'sample')}:")
print(f"  Email: {sample_train['text'][:100]}...")
print(f"  Generated label phrase: '{label_word} demonstrate empathy'")
print(f"  Model returns: a 1-2 sentence reason")

In [ ]:
def step_2_classify_with_reasons(test_row, train_with_reasons, harness, model="gpt-4o-2024-08-06"):
    '''Build a multi-turn conversation where each training example is shown
    with its generated reason, then ask the model to classify the test row.
    '''
    messages = [
        {"role": "system", "content": "You are an expert rater of workplace empathy. Below are example emails with their empathy labels and reasoning. Then classify a new email."},
    ]
    for tr in train_with_reasons:
        messages.append({"role": "user", "content": f"Email:\n###\n{tr['text']}\n###\nDoes this email demonstrate empathy? Why?"})
        messages.append({"role": "assistant", "content": f"Label: {tr['empathy']}. Reason: {tr['reason']}"})
    messages.append({"role": "user", "content": f"Email:\n###\n{test_row['text']}\n###\nDoes this email demonstrate empathy? Respond with strict JSON: {{\"label\": 0}} or {{\"label\": 1}}."})
    return messages

# This is what the PAID approach looks like in shape. The size of the messages
# array grows linearly with the training set — for ~200 training rows, the
# context is dominated by the few-shot history. PAID Team's published notebook
# uses gpt-4-0125-preview with a 128K context window; we'd use gpt-4o-2024-08-06
# the same way today.
print("Conversation length for PAID approach with all ~200 training rows: 1 system + 400 turns + 1 final = 402 messages")

## The unified-harness approach

The PAID Team built that entire pipeline as a bespoke notebook for one task. The harness in this repo abstracts the work into the EmpathyAdapter, with two key simplifications:

1. **K-of-N few-shot rather than all-N.** With 200 training rows, K=16 is enough to span the label space and tonal range while keeping context cost down by 12x.

2. **Structured outputs.** OpenAI's strict JSON schema (released August 2024, after the competition) eliminates parse failures. The PAID notebook had defensive regex parsing for cases where the model returned "Yes" or "Label: 1" instead of `{"label": 1}`; that's not needed today.

3. **Similarity-selected examples.** When K << N, the question of *which* examples to pick matters. Random gives you K examples that span the training set; similarity-by-cosine gives you K examples that look like the test row. For empathy, similarity wins by a couple of accuracy points (see KNOWN_LANDMINES.md Landmine 4).

We can skip the Step 1 reason-generation entirely. With 2026 models, the in-context labeling signal is strong enough that auto-generated reasoning doesn't add much. (If you wanted to add it back, the adapter would just need an extra field on each training example.)

In [ ]:
adapter = EmpathyAdapter()
print(f"Adapter: {adapter.task_name}")
print(f"K few-shot examples: {adapter.k_examples}")
print(f"Response format: strict JSON schema with label ∈ {{0, 1}}")
print()

# Show what one full request looks like
sample_examples = empathy_train[:3]  # 3 instead of 16 for display
sample_test_row = {"text": "Hi Jonathan, I noticed the reports are running long. Want to grab coffee and talk about it?"}
messages = adapter.build_messages(sample_test_row, sample_examples)
print(f"Number of messages: {len(messages)}")
print(f"\nSystem prompt: {messages[0]['content']}")
print(f"\nFirst user turn: {messages[1]['content'][:200]}...")
print(f"\nFirst assistant turn: {messages[2]['content']}")
print(f"\nFinal user turn: {messages[-1]['content'][:200]}...")

## Akben's Elo trick — the move that won this task

Akben's empathy approach ensembled three things and majority-voted. Two of them were standard (label-learning few-shot, text-completion N-shot with multiple base models). The third was the genuinely creative move: **Elo rating via pairwise comparison.**

The setup:
1. Sample two semantically similar emails from the test set.
2. Ask GPT-4: "Which feedback would you prefer? Pick the one that makes you feel motivated to work harder and understood."
3. Treat this as a 1v1 chess match. Update Elo ratings using the standard formula.
4. Repeat N times across the test set.
5. Convert final Elo ratings to probabilities, then threshold to binary.

Why this works: asking the model "is this empathetic?" is harder than asking "which of these two is more empathetic?" The pairwise framing leverages relative judgment, which is more reliable than absolute. It's the same principle behind LMSYS Chatbot Arena.

Below is a reconstruction. We don't actually run it against test data here, but the implementation is real.

In [ ]:
import math
import random

def elo_update(rating_a, rating_b, score_a, k=16):
    '''Standard Elo update. score_a is 1 if A won, 0 if B won.'''
    expected_a = 1 / (1 + 10 ** ((rating_b - rating_a) / 400))
    return rating_a + k * (score_a - expected_a)


def elo_pairwise_pass(test_rows, harness, model="gpt-4o-2024-08-06", n_rounds=500, seed=42):
    '''Akben's Elo approach. Returns a dict of row_id -> Elo rating.'''
    ratings = {r["_id"]: 1000.0 for r in test_rows}
    rng = random.Random(seed)
    by_id = {r["_id"]: r for r in test_rows}
    ids = list(ratings.keys())
    
    for _ in range(n_rounds):
        a_id, b_id = rng.sample(ids, 2)
        row_a, row_b = by_id[a_id], by_id[b_id]
        prompt = (
            "You are comparing two pieces of workplace feedback. Pick the one that "
            "makes you feel motivated to work harder and understood — the more "
            "empathetic of the two. Return only 'A' or 'B'.\n\n"
            f"A: {row_a['text']}\n\n"
            f"B: {row_b['text']}"
        )
        # spec = CallSpec(messages=[{"role": "user", "content": prompt}], model=model, temperature=0.0)
        # winner = harness.call(spec).strip().upper()
        # if winner not in ("A", "B"): continue
        # score_a = 1.0 if winner == "A" else 0.0
        score_a = 0.5  # placeholder; real implementation calls the API
        ratings[a_id] = elo_update(ratings[a_id], ratings[b_id], score_a)
        ratings[b_id] = elo_update(ratings[b_id], ratings[a_id], 1.0 - score_a)
    return ratings


# To convert Elo ratings to binary labels, you'd threshold at the median (or
# at a value calibrated against the training set's positive class rate). The
# nice property of the Elo approach is that it gives you ranked confidence —
# the strongest "definitely empathetic" emails sit far above the threshold,
# and you can use that for ensemble weighting downstream.

print("Elo update for 1000 vs 1000, A wins:", round(elo_update(1000, 1000, 1.0), 2))
print("Elo update for 1100 vs 900, B wins (upset):", round(elo_update(1100, 900, 0.0), 2))
print("Elo update for 1100 vs 900, A wins (expected):", round(elo_update(1100, 900, 1.0), 2))

## Self-consistency wrapper

The harness has a built-in wrapper for Akben-style self-consistency: run the same call N times with temperature > 0 and take the mode of the predictions. For empathy, this is a free ~1-2 points of accuracy with N=5.

The wrapper is opt-in via `harness.call_consistent`. For the empathy adapter:

In [ ]:
# Demonstrate self-consistency at the call level
def empathy_predict_consistent(row, examples, harness, n=5):
    adapter = EmpathyAdapter()
    messages = adapter.build_messages(row, examples)
    spec = CallSpec(
        messages=messages,
        model="gpt-4o-2024-08-06",
        temperature=0.7,  # need non-zero for diversity
        response_format=adapter.response_format(),
    )
    # text = harness.call_consistent(spec, n=n, reduce=mode_reducer)
    # return adapter.parse(text, row)
    print(f"Would issue {n} parallel-equivalent calls at T=0.7, take the mode of the parsed labels.")
    print(f"Expected cost: {n}x base; expected gain: 1-2 accuracy points if T-0 confidence is poor.")
    return None

empathy_predict_consistent({"text": "..."}, [], None, n=5)

## End-to-end run (if data is present)

Below is the single cell that runs the full pipeline. If the input files aren't in `data/`, it prints a message and stops. If they are, it runs and reports accuracy.

For the test split, this is roughly 50 API calls (the dev/test labels file has 70 empathy rows in dev; the test split has 60 rows of empathy according to the label file).

In [ ]:
from src.run import run_task

# Dev split — for iteration. Use --row-id on the CLI to debug single rows.
result = run_task(
    task="empathy",
    split="dev",
    model="gpt-4o-2024-08-06",
    self_consistency=1,  # set to 5 for Akben-style consistency
    output_path=None,
    row_id=None,
    similarity_examples=True,
)
if result["status"] == "ok":
    print(f"Empathy dev: n={result['n']}, accuracy={result.get('score', 'n/a'):.4f}")
else:
    print(f"Status: {result['status']}")
    print(f"Message: {result.get('message')}")

## Discussion — what generalizes from this task

Three takeaways from the four winners and this reconstruction:

1. **Reasoning supervision is cheap and good.** PAID's auto-reasons trick adds one cached pass over the training set; the inference-time prompt is the same. If you have budget for it, do it.

2. **Pairwise comparison beats absolute judgment for noisy human labels.** Akben's Elo idea is the genuinely new move from this competition. Generalizes to any task where you care about ranking more than calibration.

3. **Self-consistency is a free 1-2 points whenever the base model is uncertain.** Costs N× API calls. If the cost matters, run N=3 instead of N=5; the marginal gain from 3→5 is small.

What didn't work and shouldn't be repeated:

- SetFit/prompt tuning (Wonderlic) on a noisy label distribution: fine-tuning amplifies label noise.
- Sub-dimension decomposition (Hungry Llama) for empathy specifically: the seven dimensions are correlated enough that you don't get much from the decomposition, just more noise.